In [ ]:
import os
import glob
import numpy as np
import rasterio

def calculate_cv(data_stack):
    """
    To calculate Coefficient of Variation (std / mean) along the time axis (axis=0).
    Ignores NaN values which represent masked water or nodata pixels.
    """
    with np.errstate(divide='ignore', invalid='ignore'):
        mean_val = np.nanmean(data_stack, axis=0)
        std_val = np.nanstd(data_stack, axis=0)
        cv = std_val / mean_val
    return cv

def main():
    # Folder containing the Landsat images (change as needed in Colab)
    image_dir = '/content/images'
    
    print(f"Searching for Landsat 8/9 files in: {image_dir}")
    
    # -------------------------------------------------------------------------
    # 1. FILE PATTERN CONFIGURATION
    # -------------------------------------------------------------------------
    red_files = sorted(glob.glob(os.path.join(image_dir, '*_B4.tif')))   # Band 4 (Red)
    nir_files = sorted(glob.glob(os.path.join(image_dir, '*_B5.tif')))   # Band 5 (NIR)
    lst_files = sorted(glob.glob(os.path.join(image_dir, '*_B10.tif')))  # Band 10 (Thermal/LST)
    mask_files = sorted(glob.glob(os.path.join(image_dir, '*_mask.tif'))) # Water Mask
    
    if not red_files:
        print("Error: No files found. Please check your image directory and file suffixes.")
        return
        
    if not (len(red_files) == len(nir_files) == len(lst_files) == len(mask_files)):
        print("Warning: Mismatched number of files between Red, NIR, LST, and Mask bands!")

    red_stack = []
    nir_stack = []
    lst_stack = []

    # Extracting metadata from the first image to calculate coordinates later
    with rasterio.open(red_files[0]) as src:
        transform = src.transform

    print(f"Processing {len(red_files)} temporal scenes...")

    # -------------------------------------------------------------------------
    # 2. READ & MASK DATA
    # -------------------------------------------------------------------------
    for i in range(len(red_files)):
        with rasterio.open(red_files[i]) as src_red, \
             rasterio.open(nir_files[i]) as src_nir, \
             rasterio.open(lst_files[i]) as src_lst, \
             rasterio.open(mask_files[i]) as src_mask:
            
            # Read bands as floats to support NaN assignment
            red = src_red.read(1).astype(np.float32)
            nir = src_nir.read(1).astype(np.float32)
            lst = src_lst.read(1).astype(np.float32)
            
            # Read mask (assuming 1 = Water, 0 = Land - adjust if your mask is inverted!)
            mask = src_mask.read(1)

            # Convert 0/nodata values to NaN
            red[red == 0] = np.nan
            nir[nir == 0] = np.nan
            lst[lst == 0] = np.nan

            # Apply water mask: Set water pixels to NaN so they are ignored in land math
            water_condition = (mask == 1) 
            red[water_condition] = np.nan
            nir[water_condition] = np.nan
            lst[water_condition] = np.nan

            red_stack.append(red)
            nir_stack.append(nir)
            lst_stack.append(lst)

    # Convert lists to 3D numpy arrays: Dimensions -> (time, rows, cols)
    red_stack = np.array(red_stack)
    nir_stack = np.array(nir_stack)
    lst_stack = np.array(lst_stack)

    # -------------------------------------------------------------------------
    # 3. CALCULATE COEFFICIENT OF VARIATION (CV)
    # -------------------------------------------------------------------------
    print("Calculating temporal Coefficient of Variation (CV) for all land pixels...")
    cv_red = calculate_cv(red_stack)
    cv_nir = calculate_cv(nir_stack)
    cv_lst = calculate_cv(lst_stack)

    # Combine Optical CVs
    cv_optical_combined = (cv_red + cv_nir)

    # -------------------------------------------------------------------------
    # 4. PINPOINT MOST STABLE (LOWEST CV) PIXELS
    # -------------------------------------------------------------------------
    print("\n========================================================")
    print("RESULTS: MOST STABLE LAND PIXELS")
    print("========================================================")
    
    # --- A. OPTICAL BANDS (Red + NIR) ---
    if np.all(np.isnan(cv_optical_combined)):
        print("Error: All optical pixels are masked out as water/nodata.")
    else:
        min_cv_opt = np.nanmin(cv_optical_combined)
        # Find the 2D array coordinates (Row, Col) of the lowest CV pixel
        idx_opt = np.unravel_index(np.nanargmin(cv_optical_combined), cv_optical_combined.shape)
        # Convert array Row/Col to Geographic Longitude/Latitude
        lon_opt, lat_opt = rasterio.transform.xy(transform, idx_opt[0], idx_opt[1])
        
        print("\n➤ COMBINED OPTICAL (Red & NIR):")
        print(f"   Lowest CV Value : {min_cv_opt:.6f}")
        print(f"   Image (Row, Col): ({idx_opt[0]}, {idx_opt[1]})")
        print(f"   Geographic Coord: Longitude {lon_opt:.6f}, Latitude {lat_opt:.6f}")

    # --- B. THERMAL BAND (LST / Band 10) ---
    if np.all(np.isnan(cv_lst)):
        print("\nError: All LST pixels are masked out as water/nodata.")
    else:
        min_cv_lst = np.nanmin(cv_lst)
        idx_lst = np.unravel_index(np.nanargmin(cv_lst), cv_lst.shape)
        lon_lst, lat_lst = rasterio.transform.xy(transform, idx_lst[0], idx_lst[1])
        
        print("\n➤ THERMAL LST (Band 10):")
        print(f"   Lowest CV Value : {min_cv_lst:.6f}")
        print(f"   Image (Row, Col): ({idx_lst[0]}, {idx_lst[1]})")
        print(f"   Geographic Coord: Longitude {lon_lst:.6f}, Latitude {lat_lst:.6f}")

if __name__ == "__main__":
    main()
